<a href="https://colab.research.google.com/github/Rogerio-mack/IMT_CD_2026/blob/main/IMT_CD_L1L2_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de Regularização (Ridge e Lasso)

Raramente se usa a Regressão Linear pura em produção, ou casos reais. A maioria dos cientistas de dados emprega Ridge (L2) ou Lasso (L1) para evitar *overfitting*, tratar centenas ou milhares de variáveis (Lasso) ou muitas variáveis correlacionadas (Ridge, para multicolinearidade).

Os modelos de de Regularização adicionam uma penalidade baseada no tamanho dos coeficientes $beta$ da regressão.


## Ridge Regression (L2)

$$\min_{\beta} \left\{ \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^{p} \beta_j^2 \right\}$$


Encolhe os coeficientes para perto de zero, mas nunca os zera. Ideal para modelos com muitas variáveis mas que maioria das variáveis preditoras importa.

## Lasso Regression (L1)

$$\min_{\beta} \left\{ \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^{p} |\beta_j| \right\}$$


Encolhe os coeficientes e pode zerar vários deles. Por isso é ideal para modelos com centenas ou milhares de variáveis, mas muitas variáveis irrelevantes ou redundantes.

## Elastic Net

É um mix dos modelos Ridge e Lasso.

$$\min_{\beta} \left\{ \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda_1 \sum_{j=1}^{p} |\beta_j| + \lambda_2 \sum_{j=1}^{p} \beta_j^2 \right\}$$


$$\min_{\beta} \left\{ \frac{1}{2n} \sum_{i=1}^{n} \left( y_i - X_i\beta \right)^2 + \lambda \left( \alpha \sum_{j=1}^{p} |\beta_j| + \frac{1-\alpha}{2} \sum_{j=1}^{p} \beta_j^2 \right) \right\}
$$

## imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [ ]:
df = sns.load_dataset('penguins')
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male


# Hot & Label encode



In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['island', 'species', 'sex']

encoder = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
encoded_data = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols))

df = df.drop(columns=categorical_cols)
df = pd.concat([df, encoded_df], axis=1)

display(df.head())

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Dream,island_Torgersen,species_Chinstrap,species_Gentoo,sex_Male
0,39.1,18.7,181.0,3750.0,0.0,1.0,0.0,0.0,1.0
1,39.5,17.4,186.0,3800.0,0.0,1.0,0.0,0.0,0.0
2,40.3,18.0,195.0,3250.0,0.0,1.0,0.0,0.0,0.0
3,36.7,19.3,193.0,3450.0,0.0,1.0,0.0,0.0,0.0
4,39.3,20.6,190.0,3650.0,0.0,1.0,0.0,0.0,1.0


# 3. Scale

**ORBRIGATÓRIO** para modelos de Regularização!

In [ ]:
df.iloc[:,0:4]

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,36.7,19.3,193.0,3450.0
4,39.3,20.6,190.0,3650.0
...,...,...,...,...
328,47.2,13.7,214.0,4925.0
329,46.8,14.3,215.0,4850.0
330,50.4,15.7,222.0,5750.0
331,45.2,14.8,212.0,5200.0


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df.iloc[:,0:4] = scaler.fit_transform(df.iloc[:,0:4])

df.head()


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Dream,island_Torgersen,species_Chinstrap,species_Gentoo,sex_Male
0,-0.896042,0.780732,-1.426752,-0.568475,0.0,1.0,0.0,0.0,1.0
1,-0.822788,0.119584,-1.069474,-0.506286,0.0,1.0,0.0,0.0,0.0
2,-0.676280,0.424729,-0.426373,-1.190361,0.0,1.0,0.0,0.0,0.0
3,-1.335566,1.085877,-0.569284,-0.941606,0.0,1.0,0.0,0.0,0.0
4,-0.859415,1.747026,-0.783651,-0.692852,0.0,1.0,0.0,0.0,1.0


# Models

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.linear_model import HuberRegressor, RANSACRegressor, TheilSenRegressor

X = df.drop(columns='body_mass_g')
y = df['body_mass_g']

for regression_model in [LinearRegression(),
                         Ridge(alpha=1.0),
                         Lasso(alpha=0.1),
                         HuberRegressor(),
                         RANSACRegressor(random_state=42),
                         TheilSenRegressor(random_state=42),
                         ElasticNet(alpha=0.5)]:
  result = regression_model.fit(X, y)
  print(f'Model {regression_model} score = {result.score(X, y):.4f}')





Model LinearRegression() score = 0.8752
Model Ridge() score = 0.8747
Model Lasso(alpha=0.1) score = 0.7633
Model HuberRegressor() score = 0.8747
Model RANSACRegressor(random_state=42) score = 0.8661
Model TheilSenRegressor(random_state=42) score = 0.8694
Model ElasticNet(alpha=0.5) score = 0.6237
